In [144]:
import pandas as pd
import numpy as  np

In [145]:
df=pd.read_csv("/content/Retail Sales Performance Analysis using Python (Pandas & NumPy).csv")


In [146]:
df.head()

,Order_ID,Product,Category,Price,Quantity,Discount,City,Revenue,Profit
0,1,Bag,Accessories,12566,4,0.01,Chennai,50264,7036.96
1,2,Laptop,Accessories,10361,2,0.04,Hyderabad,20722,2279.42
2,3,Watch,Accessories,6174,1,0.03,Mumbai,6174,740.88
3,4,Laptop,Fashion,31830,3,0.01,Mumbai,95490,13368.60
4,5,Laptop,Electronics,47055,4,0.21,Bangalore,188220,-11293.20


In [147]:
# EDA

df.isnull().sum()
df.info()
df.tail()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Order_ID  100000 non-null  int64  
 1   Product   100000 non-null  object 
 2   Category  100000 non-null  object 
 3   Price     100000 non-null  int64  
 4   Quantity  100000 non-null  int64  
 5   Discount  100000 non-null  float64
 6   City      100000 non-null  object 
 7   Revenue   100000 non-null  int64  
 8   Profit    100000 non-null  float64
dtypes: float64(2), int64(4), object(3)
memory usage: 6.9+ MB


,Order_ID,Price,Quantity,Discount,Revenue,Profit
count,100000.000000,100000.000000,100000.000000,100000.00000,100000.000000,100000.000000
mean,50000.500000,25290.641430,2.498800,0.14963,63250.026080,11.330137
std,28867.657797,14299.292839,1.118093,0.08680,48396.261137,6908.922512
min,1.000000,500.000000,1.000000,0.00000,503.000000,-29950.200000
25%,25000.750000,12906.000000,1.000000,0.07000,24628.000000,-2980.470000
50%,50000.500000,25334.000000,3.000000,0.15000,48631.000000,0.000000
75%,75000.250000,37660.500000,3.000000,0.22000,92789.500000,3025.260000
max,100000.000000,49998.000000,4.000000,0.30000,199992.000000,29992.200000


Q.1: Profitability Problem :
The company is generating sales but does not know which categories and products are actually profitable and which are causing losses.

In [148]:
#Category profit margin '
category = df.groupby("Category")[["Revenue","Profit"]].sum()
category
category["Profit_Margin_%"] = (
    (category["Profit"] / category["Revenue"]) * 100).round(2)
category

,Revenue,Profit,Profit_Margin_%
Category,,,
Accessories,2114488654,268103.25,0.01
Electronics,2105116491,339632.94,0.02
Fashion,2105397463,525277.50,0.02


Q2.
Discount Strategy Problem :
Discounts are being applied without a clear understanding of their impact on profit, which may be reducing overall profitability.


In [149]:
df.columns


Index(['Order_ID', 'Product', 'Category', 'Price', 'Quantity', 'Discount',
       'City', 'Revenue', 'Profit'],
      dtype='object')

In [150]:
#Discount vs Profit

discount_analysis = df.groupby("Discount")["Profit"].mean().sort_index()
discount_analysis


,Profit
Discount,
0.00,9482.130796
0.01,8698.252095
0.02,8308.209851
0.03,7669.939280
0.04,6882.631791
0.05,6232.674578
0.06,5639.623461
0.07,5086.991599
0.08,4357.414827


In [151]:
#Discount Buckets

df["Discount_Level"] = pd.cut(df["Discount"],bins=[0, 0.1, 0.2, 0.3],labels=["Low (0-10%)", "Medium (10-20%)", "High (20-30%)"])


In [152]:
df

,Order_ID,Product,Category,Price,Quantity,Discount,City,Revenue,Profit,Discount_Level
0,1,Bag,Accessories,12566,4,0.01,Chennai,50264,7036.96,Low (0-10%)
1,2,Laptop,Accessories,10361,2,0.04,Hyderabad,20722,2279.42,Low (0-10%)
2,3,Watch,Accessories,6174,1,0.03,Mumbai,6174,740.88,Low (0-10%)
3,4,Laptop,Fashion,31830,3,0.01,Mumbai,95490,13368.60,Low (0-10%)
4,5,Laptop,Electronics,47055,4,0.21,Bangalore,188220,-11293.20,High (20-30%)
...,...,...,...,...,...,...,...,...,...,...
99995,99996,Shoes,Electronics,47372,1,0.05,Delhi,47372,4737.20,Low (0-10%)
99996,99997,Shirt,Fashion,5879,3,0.24,Delhi,17637,-1587.33,High (20-30%)
99997,99998,Laptop,Fashion,4778,1,0.29,Mumbai,4778,-668.92,High (20-30%)
99998,99999,Bag,Accessories,38294,4,0.27,Bangalore,153176,-18381.12,High (20-30%)


In [153]:
#Compare Profit by Discount Level

discount_bucket_analysis = df.groupby("Discount_Level").agg({
    "Profit": "mean",
    "Revenue": "mean",
    "Order_ID": "count"
})

discount_bucket_analysis

/tmp/ipykernel_2811/1309617136.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  discount_bucket_analysis = df.groupby("Discount_Level").agg({


,Profit,Revenue,Order_ID
Discount_Level,,,
Low (0-10%),6000.242047,63066.831067,33392
Medium (10-20%),-306.643530,63153.116074,33315
High (20-30%),-6510.131954,63548.131785,31559


In [156]:
#Check Loss Orders

loss_orders = df[df["Profit"] < 0]

loss_percentage = (len(loss_orders) / len(df)) * 100

print ("loss per order ",loss_percentage)

loss per order  48.252


In [161]:
#Loss from High Discount Only

high_discount_loss = df[df["Discount"] > 0.1]["Profit"].sum()


print("Loss due to high discount:", round(high_discount_loss, 2))

Loss due to high discount: -215669083.54


Q3:  Product Performance Problem :
The company lacks clarity on which products are top-performing in terms of revenue and profit and which products are underperforming.



In [162]:
df.columns

Index(['Order_ID', 'Product', 'Category', 'Price', 'Quantity', 'Discount',
       'City', 'Revenue', 'Profit', 'Discount_Level'],
      dtype='object')

In [165]:
product_performance = df.groupby("Product") [["Revenue", "Profit"]].sum()

In [166]:
product_performance

,Revenue,Profit
Product,,
Bag,1258805949,240313.06
Laptop,1274120068,1605067.21
Shirt,1281833916,-41872.85
Shoes,1272033523,-629831.27
Watch,1238209152,-40662.46



Q4.City Performance Problem ::
Sales and profit vary across cities, but the company does not know which locations are performing well and which need improvement.


In [167]:
df.columns

Index(['Order_ID', 'Product', 'Category', 'Price', 'Quantity', 'Discount',
       'City', 'Revenue', 'Profit', 'Discount_Level'],
      dtype='object')

In [169]:
city_analysis  =df.groupby("City")[["Revenue", "Profit"]].sum()

city_analysis

,Revenue,Profit
City,,
Bangalore,1260108327,77531.48
Chennai,1287074391,26016.14
Delhi,1266347081,1115796.49
Hyderabad,1265504272,275205.13
Mumbai,1245968537,-361535.55


Q5. Profit Margin Problem :
The company focuses on revenue but does not track profit margins effectively across categories and products.


In [170]:
df.columns

Index(['Order_ID', 'Product', 'Category', 'Price', 'Quantity', 'Discount',
       'City', 'Revenue', 'Profit', 'Discount_Level'],
      dtype='object')

In [183]:
# Group by Category + Product

product_cat = df.groupby(["Category", "Product"])[["Revenue", "Profit"]].sum()

product_cat

Revenue      Profit
Category    Product                       
Accessories Bag      415573544   365901.34
            Laptop   426921413   325053.26
            Shirt    425833562   789762.23
            Shoes    430465779  -647824.21
            Watch    415694356  -564789.37
Electronics Bag      418403242 -1065665.26
            Laptop   426482826  1190849.07
            Shirt    420022313  -846622.97
            Shoes    431008628   646113.64
            Watch    409199482   414958.46
Fashion     Bag      424829163   940076.98
            Laptop   420715829    89164.88
            Shirt    435978041    14987.89
            Shoes    410559116  -628120.70
            Watch    413315314   109168.45

Q6.
High Discount Risk Problem ::
Some transactions may have high discounts that lead to low or negative profit, but these are not identified or controlled.


In [184]:
df.columns

Index(['Order_ID', 'Product', 'Category', 'Price', 'Quantity', 'Discount',
       'City', 'Revenue', 'Profit', 'Discount_Level'],
      dtype='object')

In [190]:
#Transaction-Level Table (Include Discount)
transactions = df.groupby("Order_ID").agg({
    "Revenue": "sum",
    "Profit": "sum",
    "Discount": "mean"
})

transactions

,Revenue,Profit,Discount
Order_ID,,,
1,50264,7036.96,0.01
2,20722,2279.42,0.04
3,6174,740.88,0.03
4,95490,13368.60,0.01
5,188220,-11293.20,0.21
...,...,...,...
99996,47372,4737.20,0.05
99997,17637,-1587.33,0.24
99998,4778,-668.92,0.29


In [192]:
#Identify Loss-Making Transactions

loss_txn = transactions[transactions["Profit"] < 0]

loss_txn


,Revenue,Profit,Discount
Order_ID,,,
5,188220,-11293.20,0.21
8,1032,-61.92,0.21
10,16890,-844.50,0.20
15,44202,-5746.26,0.28
16,30580,-4587.00,0.30
...,...,...,...
99987,95373,-1907.46,0.17
99993,103434,-12412.08,0.27
99997,17637,-1587.33,0.24


In [194]:
#Identify High Discount Transactions (>10%)

high_discount_txn = transactions[transactions["Discount"] > 0.1]

high_discount_txn

,Revenue,Profit,Discount
Order_ID,,,
5,188220,-11293.20,0.21
8,1032,-61.92,0.21
9,87976,0.00,0.15
10,16890,-844.50,0.20
15,44202,-5746.26,0.28
...,...,...,...
99992,188532,3770.64,0.13
99993,103434,-12412.08,0.27
99997,17637,-1587.33,0.24
